# 01 — Análise e Anotação da Base

Este notebook analisa a base `tv.ibyte.jsonl`, define as entidades NER e gera o arquivo de anotações.

In [1]:
import json, re, random
import pandas as pd
from collections import Counter

PATH="/content/tv.ibyte.jsonl"
records=[]
with open(PATH,encoding="utf-8") as f:
    for line in f:
        if line.strip(): records.append(json.loads(line))

print("Registros:",len(records))
print("Chaves do primeiro registro:", list(records[0].keys()) if isinstance(records[0],dict) else type(records[0]))

Registros: 818
Chaves do primeiro registro: ['text', 'entities']


In [2]:
def get_text(r):
    if isinstance(r,str): return r
    if isinstance(r,dict):
        for k in ["Product","product","title","Title","name","Name","text"]:
            if r.get(k) is not None: return str(r[k])
    return None

texts=[get_text(r) for r in records if get_text(r) is not None]
print("Títulos:",len(texts))
for t in texts[:10]: print("-",t)

Títulos: 818
- smart tv led 32 samsung 32t4300 hd wifi plataforma tizen hdmi usb preta
- fire tv amazon stick lite 2 geracao com alexa
- fire tv stick com controle remoto por voz com alexa inclui comandos de tv streaming em full hd
- smart tv samsung 50 uhd 4k 50au7700 processador crystal 4k tela sem limites visual livre de cabos alexa built in controle unico
- smart tv led 55 4k ultra hd toshiba tb011m 55c350ls
- smart tv led 32 hd hq conversor digital hqstv32nk 66523
- smart tv 50 qled 4k samsung 50q60b tizen 3 hdmi 2 usb wi fi
- smart tv samsung 55 4k qled the frame 2022 55ls03b qn55ls03bagxzd
- smart tv led multilaser 24 hd wifi integrado hdmi usb tl040
- smart tv 42 lg oled 4k 120h g sync freesync hdmi 2 1 inteligencia artificial thinq alexa oled42c2psa


## TAGs

Para os títulos desta base, o experimento utiliza:

| TAG | Significado |
|---|---|
| `PRODUTO` | Nome/descrição principal do televisor |
| `MARCA` | Fabricante/marca |
| `TAMANHO` | Dimensão da tela |
| `TECNOLOGIA` | Tecnologia de tela |
| `RESOLUCAO` | Resolução |
| `RECURSO` | Plataforma/conectividade/recurso |
| `REFERENCIA` | Código/modelo comercial identificado por padrão alfanumérico |

As TAGs devem ser revisadas pelo grupo antes da anotação final.

In [3]:
BRANDS=["Samsung","LG","TCL","Philips","AOC","Sony","Panasonic","Hisense","Britânia","Philco","Multilaser","Semp","Epson","Lenovo","Apple"]
SIZE_RE=re.compile(r"\b(?:1[5-9]|[2-9]\d|[1-9]\d{2})(?:[.,]\d+)?\s*(?:\"|polegadas?|pol\.?)\b",re.I)
RES_RE=re.compile(r"\b(?:4K|8K|Full\s*HD|HD|UHD|QHD|2K)\b",re.I)
SCREEN_RE=re.compile(r"\b(?:LED|OLED|QLED|Mini[- ]LED|NanoCell|LCD|PLASMA)\b",re.I)
REF_RE=re.compile(r"\b[A-Z0-9]+(?:[-/.][A-Z0-9]+)+\b|\b[A-Z]{1,5}\d{2,}[A-Z0-9-]*\b",re.I)
FEATURE_RE=re.compile(r"\b(?:Smart\s*TV|Wi[- ]?Fi|Bluetooth|Android\s*TV|Google\s*TV|Roku\s*TV|Fire\s*TV)\b",re.I)

In [4]:
def annotate(text):
    entities=[]; occupied=[]
    def add(s,e,label):
        if not any(not(e<=a or s>=b) for a,b,_ in occupied):
            occupied.append((s,e,label)); entities.append([s,e,label])

    for b in BRANDS:
        for m in re.finditer(rf"(?<!\w){re.escape(b)}(?!\w)",text,re.I): add(m.start(),m.end(),"MARCA")
    for m in SCREEN_RE.finditer(text): add(m.start(),m.end(),"TECNOLOGIA")
    for m in SIZE_RE.finditer(text): add(m.start(),m.end(),"TAMANHO")
    for m in RES_RE.finditer(text): add(m.start(),m.end(),"RESOLUCAO")
    for m in FEATURE_RE.finditer(text): add(m.start(),m.end(),"RECURSO")
    for m in REF_RE.finditer(text):
        if any(c.isdigit() for c in m.group()) and any(c.isalpha() for c in m.group()):
            add(m.start(),m.end(),"REFERENCIA")

    patterns=[
        r"\b(?:Smart\s+TV|TV|Televisor)\b(?:\s+[A-Za-z0-9][A-Za-z0-9\-]*){0,7}",
        r"\b(?:OLED|QLED|LED|NanoCell)\b(?:\s+[A-Za-z0-9][A-Za-z0-9\-]*){0,5}"
    ]
    candidates=[]
    for p in patterns:
        candidates += [(m.start(),m.end()) for m in re.finditer(p,text,re.I)]
    for s,e in sorted(candidates,key=lambda x:x[1]-x[0],reverse=True):
        if not any(not(e<=a or s>=b) for a,b,_ in occupied):
            add(s,e,"PRODUTO"); break
    entities.sort()
    return {"text":text,"entities":entities}

annotations=[annotate(t) for t in texts]
print("Documentos:",len(annotations))
print("Exemplo:",annotations[0])

Documentos: 818
Exemplo: {'text': 'smart tv led 32 samsung 32t4300 hd wifi plataforma tizen hdmi usb preta', 'entities': [[0, 8, 'RECURSO'], [9, 12, 'TECNOLOGIA'], [16, 23, 'MARCA'], [32, 34, 'RESOLUCAO'], [35, 39, 'RECURSO']]}


In [5]:
counts=Counter(e[2] for x in annotations for e in x["entities"])
pd.DataFrame(sorted(counts.items()),columns=["TAG","Quantidade"])

,TAG,Quantidade
0,MARCA,649
1,PRODUTO,20
2,RECURSO,1003
3,REFERENCIA,371
4,RESOLUCAO,881
5,TAMANHO,60
6,TECNOLOGIA,438


In [6]:
for x in annotations[:10]:
    print("\n",x["text"])
    for s,e,l in x["entities"]:
        print(f"  {x['text'][s:e]!r} -> {l}")


 smart tv led 32 samsung 32t4300 hd wifi plataforma tizen hdmi usb preta
  'smart tv' -> RECURSO
  'led' -> TECNOLOGIA
  'samsung' -> MARCA
  'hd' -> RESOLUCAO
  'wifi' -> RECURSO

 fire tv amazon stick lite 2 geracao com alexa
  'fire tv' -> RECURSO

 fire tv stick com controle remoto por voz com alexa inclui comandos de tv streaming em full hd
  'fire tv' -> RECURSO
  'full hd' -> RESOLUCAO

 smart tv samsung 50 uhd 4k 50au7700 processador crystal 4k tela sem limites visual livre de cabos alexa built in controle unico
  'smart tv' -> RECURSO
  'samsung' -> MARCA
  'uhd' -> RESOLUCAO
  '4k' -> RESOLUCAO
  '4k' -> RESOLUCAO

 smart tv led 55 4k ultra hd toshiba tb011m 55c350ls
  'smart tv' -> RECURSO
  'led' -> TECNOLOGIA
  '4k' -> RESOLUCAO
  'hd' -> RESOLUCAO
  'tb011m' -> REFERENCIA

 smart tv led 32 hd hq conversor digital hqstv32nk 66523
  'smart tv' -> RECURSO
  'led' -> TECNOLOGIA
  'hd' -> RESOLUCAO
  'hqstv32nk' -> REFERENCIA

 smart tv 50 qled 4k samsung 50q60b tizen 3 hdmi 

In [7]:
idx=list(range(len(annotations)))
random.Random(42).shuffle(idx)
cut=int(.8*len(idx))
train=[annotations[i] for i in idx[:cut]]
test=[annotations[i] for i in idx[cut:]]
print("Treinamento:",len(train),"| Teste:",len(test))

Treinamento: 654 | Teste: 164


In [8]:
import os

def save_jsonl(path,data):
    # Ensure the directory exists
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path,"w",encoding="utf-8") as f:
        for x in data: f.write(json.dumps(x,ensure_ascii=False)+"\n")
save_jsonl("../data/annotations/tv.jsonl",annotations)
save_jsonl("../data/processed/tv_train.jsonl",train)
save_jsonl("../data/processed/tv_test.jsonl",test)
print("Arquivos salvos.")

Arquivos salvos.


**Limitação:** as anotações deste primeiro experimento são semiautomáticas, baseadas em regras. A revisão humana no Doccano é recomendada antes da avaliação final.

# Nova seção